# AML/TM Databricks One-Stop Learning Notebook

Run this notebook top to bottom in Azure Databricks, Fabric Spark notebooks, or a Jupyter environment with PySpark. It is the single runnable notebook for the Spark/Databricks learning path: platform mental model, widgets, Spark SQL, PySpark, DQ checks, reconciliation, Delta-style persistence, Databricks SQL outputs, Lakeflow/Jobs thinking, ML feature readiness, focused PySpark basics, focused Spark SQL basics, and the tech-stack SQL-versus-PySpark micro-lab.

## What This Notebook Covers

This notebook uses a tiny public-safe AML/TM modernization case study:

```text
legacy extracts -> bronze raw data -> silver standardized data -> gold rule input
               -> DQ exceptions + reconciliation -> alerts + supporting transactions
               -> Databricks SQL / BI views -> job evidence -> feature-ready analytics
```

The data is synthetic. No private customer, bank, screenshot, credential, or proprietary rule logic is used.

## Step 0 - Databricks-Style Parameters and Helpers

Databricks notebooks often use widgets for job parameters. This cell uses widgets when `dbutils` exists and falls back to plain Python values outside Databricks.

**Code-cell explanation:**

- Purpose: create the shared runtime context used by every later cell.
- Inputs: optional Databricks widgets for `processing_month` and `rule_threshold_cad`; outside Databricks, the cell falls back to public-safe defaults.
- Walkthrough: imports Spark helpers, creates `SparkSession`, detects `dbutils`, calculates `month_start`, `month_end`, and `batch_id`, and defines reusable assertion/display helpers.
- Before running: predict `processing_month=2022-06`, `month_end=2022-07-01`, and `rule_threshold_cad=100.0` unless widgets override them.
- Failure meaning: if this cell fails, the notebook does not have a working Spark runtime or the widget fallback path needs attention.

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("aml-databricks-one-stop-learning").getOrCreate()

def get_dbutils_or_none():
    try:
        return dbutils
    except NameError:
        return None

dbutils_ref = get_dbutils_or_none()

if dbutils_ref is not None:
    dbutils_ref.widgets.text("processing_month", "2022-06")
    dbutils_ref.widgets.text("rule_threshold_cad", "100")
    processing_month = dbutils_ref.widgets.get("processing_month")
    rule_threshold_cad = float(dbutils_ref.widgets.get("rule_threshold_cad"))
else:
    processing_month = "2022-06"
    rule_threshold_cad = 100.0

month_start = f"{processing_month}-01"
month_end = spark.sql(f"SELECT add_months(DATE '{month_start}', 1) AS month_end").first().month_end.isoformat()
batch_id = f"aml_tm_demo_{processing_month.replace('-', '')}"

def assert_set(name, actual_rows, expected_rows):
    actual = set(actual_rows)
    expected = set(expected_rows)
    assert actual == expected, f"{name}: expected {expected}, got {actual}"

def show_df(df, n=20):
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

print(f"processing_month={processing_month}")
print(f"month_start={month_start}, month_end={month_end}")
print(f"rule_threshold_cad={rule_threshold_cad}")
print(f"batch_id={batch_id}")

## Step 1 - Create Synthetic Source Extracts

In a real Databricks project these rows might arrive from files, ADF/Fabric Data Factory, Lakeflow Connect, or upstream exports. Here they are created in-memory so the notebook is self-contained.

**Code-cell explanation:**

- Purpose: create tiny public-safe source extracts that behave like legacy transaction, account, and country-risk feeds.
- Inputs: in-memory rows only; no private tables, files, credentials, or workspace paths.
- Walkthrough: defines explicit schemas, creates source DataFrames, and checks the expected source counts.
- Before running: predict 8 transaction rows, 4 account rows, and 3 country-risk rows.
- Failure meaning: a count mismatch means the learning dataset changed and all downstream DQ, alert, and reconciliation expectations must be reviewed.

In [ ]:
transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("account_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("amount_cad", T.StringType(), False),
    T.StructField("transaction_type", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("country_code", T.StringType(), True),
])

source_transactions = spark.createDataFrame([
    ("t1", "a1", "2022-06-01", "60.00", "WIRE", "POSTED", "IR"),
    ("t2", "a1", "2022-06-03", "50.00", "WIRE", "POSTED", "IR"),
    ("t3", "a1", "2022-06-05", "10.00", "CARD", "POSTED", "CA"),
    ("t4", "a2", "2022-06-02", "200.00", "WIRE", "POSTED", "CA"),
    ("t5", "a3", "2022-06-02", "20.00", "WIRE", "REVERSED", "IR"),
    ("t6", "a9", "2022-06-02", "80.00", "WIRE", "POSTED", "IR"),
    ("t7", "a2", "2022-07-01", "300.00", "WIRE", "POSTED", "IR"),
    ("t8", "a4", "2022-06-10", "100.00", "CASH", "POSTED", None),
], schema=transaction_schema)

accounts = spark.createDataFrame([
    ("a1", "c1", "ACTIVE", "CHECKING"),
    ("a2", "c2", "ACTIVE", "CHECKING"),
    ("a3", "c3", "ACTIVE", "SAVINGS"),
    ("a4", "c4", "CLOSED", "CHECKING"),
], ["account_id", "customer_id", "account_status", "product_type"])

country_risk = spark.createDataFrame([
    ("IR", "HIGH"),
    ("CA", "LOW"),
    ("US", "LOW"),
], ["country_code", "risk_level"])

assert source_transactions.count() == 8
assert accounts.count() == 4
assert country_risk.count() == 3
show_df(source_transactions.orderBy("transaction_id"))

## Step 2 - Bronze Layer: Preserve Raw Shape and Add Run Metadata

Bronze should preserve what arrived while adding enough metadata to replay and audit the run.

**Code-cell explanation:**

- Purpose: model a bronze table that preserves source-like values while adding lineage metadata.
- Inputs: `source_transactions` from Step 1 and the `batch_id` from Step 0.
- Walkthrough: adds batch/source/load metadata but intentionally keeps raw string dates and amounts unchanged.
- Before running: predict that row count remains 8 because bronze should not filter business records.
- Failure meaning: lost or changed rows in bronze indicate ingestion logic is doing transformation work too early.

In [ ]:
bronze_transactions = (
    source_transactions
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("source_system", F.lit("legacy_tm_extract"))
    .withColumn("ingested_at_utc", F.current_timestamp())
)

bronze_manifest = spark.createDataFrame([
    (batch_id, "transactions", bronze_transactions.count(), processing_month),
    (batch_id, "accounts", accounts.count(), processing_month),
    (batch_id, "country_risk", country_risk.count(), processing_month),
], ["batch_id", "dataset_name", "row_count", "processing_month"])

assert bronze_manifest.count() == 3
show_df(bronze_manifest.orderBy("dataset_name"))

## Step 3 - Silver Layer: Normalize Types and Standardize Keys

Silver data should be typed, standardized, and ready for DQ checks. This is where string dates become dates, numeric strings become decimals, and keys are normalized.

**Code-cell explanation:**

- Purpose: turn raw source-shaped rows into typed, standardized rows ready for DQ and rule logic.
- Inputs: `bronze_transactions`.
- Walkthrough: converts date and amount strings, trims/uppercases keys and codes, and preserves source/run metadata.
- Before running: predict row count remains 8, but data types change for `transaction_date` and `amount_cad`.
- Failure meaning: type conversion failures usually point to source-format drift, bad input data, or missing DQ rules.

In [ ]:
silver_transactions = (
    bronze_transactions
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
    .withColumn("amount_cad", F.col("amount_cad").cast("decimal(18,2)"))
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("country_code", F.upper(F.trim("country_code")))
    .withColumn("transaction_type", F.upper(F.trim("transaction_type")))
    .withColumn("status", F.upper(F.trim("status")))
)

assert silver_transactions.count() == 8
assert dict(silver_transactions.dtypes)["transaction_date"] == "date"
assert dict(silver_transactions.dtypes)["amount_cad"] == "decimal(18,2)"
show_df(silver_transactions.orderBy("transaction_id"))

## Step 4 - DQ Checks: Find Exceptions Before Rule Execution

A Databricks pipeline should not silently lose bad rows. Separate DQ exceptions from valid rule input and reconcile the counts.

**Code-cell explanation:**

- Purpose: make bad or risky rows visible before alert logic runs.
- Inputs: `silver_transactions` plus `accounts` and `country_risk` reference data.
- Walkthrough: separates required-field failures, orphan accounts, missing risk references, valid customer transactions, and a DQ summary.
- Before running: predict one orphan account transaction: `t6` has account `A9` and should not silently disappear.
- Failure meaning: if DQ exceptions are not visible, downstream alert counts can look clean while hiding control breaks.

In [ ]:
required_field_failures = silver_transactions.filter(
    F.col("transaction_id").isNull()
    | F.col("account_id").isNull()
    | F.col("transaction_date").isNull()
    | F.col("amount_cad").isNull()
)

orphan_accounts = silver_transactions.join(accounts, on="account_id", how="left_anti")

invalid_country = silver_transactions.filter(F.col("country_code").isNotNull()).join(
    country_risk,
    on="country_code",
    how="left_anti",
)

duplicate_transaction_ids = (
    silver_transactions.groupBy("transaction_id")
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
)

closed_account_transactions = (
    silver_transactions.join(accounts, on="account_id", how="inner")
    .filter(F.col("account_status") == "CLOSED")
)

dq_summary = spark.createDataFrame([
    ("required_field_failures", required_field_failures.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("invalid_country", invalid_country.count()),
    ("duplicate_transaction_ids", duplicate_transaction_ids.count()),
    ("closed_account_transactions", closed_account_transactions.count()),
], ["dq_check", "failed_row_count"])

assert required_field_failures.count() == 0
assert_set("orphan_accounts", [r.transaction_id for r in orphan_accounts.select("transaction_id").collect()], ["t6"])
assert invalid_country.count() == 0
assert duplicate_transaction_ids.count() == 0
assert_set("closed_account_transactions", [r.transaction_id for r in closed_account_transactions.select("transaction_id").collect()], ["t8"])
show_df(dq_summary.orderBy("dq_check"))

## Step 5 - Gold Rule Input: Create the Governed Rule-Ready Grain

The rule grain here is one eligible transaction joined to a valid customer/account and country-risk reference row.

**Code-cell explanation:**

- Purpose: create the exact transaction-level grain that the rule is allowed to use.
- Inputs: `valid_customer_transactions` and `country_risk`.
- Walkthrough: filters June posted wires, attaches high-risk country labels, and keeps only valid rule input rows.
- Before running: predict 3 June posted wire rows with valid accounts, then 2 high-risk rows for customer `C1`.
- Failure meaning: wrong counts here usually come from date-window mistakes, join loss, or risk-reference coverage problems.

In [ ]:
june_posted_wires = (
    silver_transactions
    .filter(F.col("status") == "POSTED")
    .filter(F.col("transaction_type") == "WIRE")
    .filter((F.col("transaction_date") >= F.lit(month_start)) & (F.col("transaction_date") < F.lit(month_end)))
)

valid_customer_tx = june_posted_wires.join(accounts, on="account_id", how="inner")

gold_rule_input = (
    valid_customer_tx.join(country_risk, on="country_code", how="left")
    .withColumn("processing_month", F.lit(processing_month))
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
)

assert june_posted_wires.count() == 4
assert valid_customer_tx.count() == 3
assert gold_rule_input.count() == 3
show_df(gold_rule_input.orderBy("transaction_id"))

## Step 6 - Spark SQL and PySpark Are Two Front Doors

Register temp views so the same rule-ready data can be queried with Spark SQL. This mirrors how Databricks teams often mix PySpark transformations and SQL validation.

**Code-cell explanation:**

- Purpose: prove that the same rule-ready grain can be aggregated with Spark SQL and PySpark.
- Inputs: `gold_rule_input` from Step 5.
- Walkthrough: registers a temp view, calculates customer totals in SQL, calculates the same totals in PySpark, then compares the row sets.
- Before running: predict one total row for `C1` with amount `110.00` and 2 supporting transactions.
- Failure meaning: if SQL and PySpark results diverge, the two implementations no longer encode the same business rule.

In [ ]:
gold_rule_input.createOrReplaceTempView("gold_rule_input")

sql_customer_totals = spark.sql(f"""
SELECT
  customer_id,
  SUM(amount_cad) AS observed_amount_cad,
  COUNT(*) AS supporting_transaction_count
FROM gold_rule_input
WHERE risk_level = 'HIGH'
GROUP BY customer_id
HAVING SUM(amount_cad) > {rule_threshold_cad}
""")

pyspark_customer_totals = (
    gold_rule_input.filter(F.col("risk_level") == "HIGH")
    .groupBy("customer_id")
    .agg(
        F.sum("amount_cad").alias("observed_amount_cad"),
        F.count("*").alias("supporting_transaction_count"),
    )
    .filter(F.col("observed_amount_cad") > F.lit(rule_threshold_cad))
)

assert sql_customer_totals.collect() == pyspark_customer_totals.collect()
show_df(sql_customer_totals)

## Step 7 - Alert Output and Supporting Transactions

An AML/TM alert should be explainable: deterministic key, rule metadata, observed value, threshold, and linked supporting transactions.

**Code-cell explanation:**

- Purpose: convert customer totals into explainable alert and support-record outputs.
- Inputs: `pyspark_customer_totals`, `gold_rule_input`, and run/rule metadata.
- Walkthrough: applies the threshold, creates deterministic alert keys, and joins the alert back to supporting transaction rows.
- Before running: predict one alert for customer `C1` and two supporting transactions: `t1` and `t2`.
- Failure meaning: alert rows without supporting rows are weak evidence; supporting rows without alert metadata are hard to audit.

In [ ]:
alerts = (
    pyspark_customer_totals
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
    .withColumn("processing_month", F.lit(processing_month))
    .withColumn("threshold_cad", F.lit(rule_threshold_cad).cast("decimal(18,2)"))
    .withColumn(
        "alert_key",
        F.sha2(F.concat_ws("|", "rule_id", "rule_version", "processing_month", "customer_id"), 256),
    )
)

supporting_transactions = (
    gold_rule_input.filter(F.col("risk_level") == "HIGH")
    .join(alerts.select("alert_key", "customer_id"), on="customer_id", how="inner")
    .select(
        "alert_key",
        "transaction_id",
        "customer_id",
        "account_id",
        "transaction_date",
        "amount_cad",
        "country_code",
        "risk_level",
    )
)

assert alerts.count() == 1
assert_set("alert customers", [r.customer_id for r in alerts.select("customer_id").collect()], ["c1"])
assert_set("supporting transaction ids", [r.transaction_id for r in supporting_transactions.select("transaction_id").collect()], ["t1", "t2"])
show_df(alerts.select("alert_key", "customer_id", "observed_amount_cad", "threshold_cad", "supporting_transaction_count"))
show_df(supporting_transactions.orderBy("transaction_id"))

## Step 8 - Reconciliation and Evidence Pack

Reconciliation is not a side report. It is evidence that rows moved as expected and that exceptions were visible.

**Code-cell explanation:**

- Purpose: produce evidence that rows moved through the pipeline as expected.
- Inputs: every major DataFrame created so far.
- Walkthrough: creates stage counts, expected counts, mismatch detection, and an evidence manifest for the run.
- Before running: predict all reconciliation mismatches are zero.
- Failure meaning: a mismatch is not automatically a code bug, but it must become a visible defect or an approved explained difference.

In [ ]:
reconciliation = spark.createDataFrame([
    ("source_transactions", source_transactions.count()),
    ("bronze_transactions", bronze_transactions.count()),
    ("silver_transactions", silver_transactions.count()),
    ("june_posted_wires", june_posted_wires.count()),
    ("valid_customer_tx", valid_customer_tx.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("gold_rule_input", gold_rule_input.count()),
    ("high_risk_supporting_tx", supporting_transactions.count()),
    ("alerts", alerts.count()),
], ["step_name", "row_count"])

expected_counts = {
    "source_transactions": 8,
    "bronze_transactions": 8,
    "silver_transactions": 8,
    "june_posted_wires": 4,
    "valid_customer_tx": 3,
    "orphan_accounts": 1,
    "gold_rule_input": 3,
    "high_risk_supporting_tx": 2,
    "alerts": 1,
}
actual_counts = {r.step_name: r.row_count for r in reconciliation.collect()}
assert actual_counts == expected_counts, f"Expected {expected_counts}, got {actual_counts}"

evidence_manifest = spark.createDataFrame([
    (batch_id, processing_month, "rule_id", "TM_HIGH_RISK_WIRE_001"),
    (batch_id, processing_month, "rule_version", "1.0.0"),
    (batch_id, processing_month, "threshold_cad", str(rule_threshold_cad)),
    (batch_id, processing_month, "alert_count", str(alerts.count())),
    (batch_id, processing_month, "dq_orphan_count", str(orphan_accounts.count())),
], ["batch_id", "processing_month", "evidence_name", "evidence_value"])

show_df(reconciliation)
show_df(evidence_manifest.orderBy("evidence_name"))

## Step 9 - Delta-Style Persistence Demo

Databricks production pipelines usually write governed Delta tables. This cell tries Delta first and falls back to Parquet if the current environment does not have Delta configured. The learning point is the same: persist curated outputs and reload them for validation.

**Code-cell explanation:**

- Purpose: show how curated outputs should be persisted and reloaded for validation.
- Inputs: `gold_rule_input`.
- Walkthrough: writes Delta when available, falls back to Parquet outside Delta-enabled environments, reloads the data, validates row counts, and cleans local fallback files.
- Before running: predict the persisted/reloaded count equals the `gold_rule_input` count.
- Failure meaning: persistence failures usually point to environment differences, missing Delta support, or unsafe output paths.

In [ ]:
if dbutils_ref is not None:
    demo_path = "dbfs:/tmp/aml_learning_for_fintech/databricks_one_stop_gold_rule_input"
    dbutils_ref.fs.rm(demo_path, True)
else:
    demo_path = "/tmp/aml_learning_for_fintech_databricks_one_stop_gold_rule_input"
    shutil.rmtree(demo_path, ignore_errors=True)

try:
    gold_rule_input.write.format("delta").mode("overwrite").save(demo_path)
    reloaded_gold_rule_input = spark.read.format("delta").load(demo_path)
    storage_format = "delta"
except Exception as delta_error:
    if dbutils_ref is not None:
        dbutils_ref.fs.rm(demo_path, True)
    else:
        shutil.rmtree(demo_path, ignore_errors=True)
    gold_rule_input.write.mode("overwrite").parquet(demo_path)
    reloaded_gold_rule_input = spark.read.parquet(demo_path)
    storage_format = f"parquet fallback after {delta_error.__class__.__name__}"

assert reloaded_gold_rule_input.count() == gold_rule_input.count()
print(f"Persisted and reloaded gold_rule_input using {storage_format} at {demo_path}")

## Step 10 - Databricks SQL / BI-Ready Views

Databricks SQL and BI tools should consume governed outputs, not raw ad hoc temp data. Here we create views that mirror alert, support, DQ, and reconciliation reporting surfaces.

**Code-cell explanation:**

- Purpose: expose governed reporting surfaces instead of forcing BI tools to query intermediate scratch DataFrames.
- Inputs: alerts, supporting transactions, DQ summary, and reconciliation outputs.
- Walkthrough: registers temp views and builds BI-style summaries for alert volume, DQ status, and reconciliation status.
- Before running: predict one alert row and zero reconciliation mismatches.
- Failure meaning: if BI views disagree with evidence tables, the dashboard layer is no longer trustworthy.

In [ ]:
alerts.createOrReplaceTempView("gold_alerts")
supporting_transactions.createOrReplaceTempView("gold_alert_supporting_transactions")
dq_summary.createOrReplaceTempView("gold_dq_summary")
reconciliation.createOrReplaceTempView("gold_reconciliation")

dashboard_summary = spark.sql("""
SELECT 'alerts' AS metric_name, COUNT(*) AS metric_value FROM gold_alerts
UNION ALL
SELECT 'supporting_transactions' AS metric_name, COUNT(*) AS metric_value FROM gold_alert_supporting_transactions
UNION ALL
SELECT 'dq_checks_with_failures' AS metric_name, COUNT(*) AS metric_value FROM gold_dq_summary WHERE failed_row_count > 0
UNION ALL
SELECT 'reconciliation_steps' AS metric_name, COUNT(*) AS metric_value FROM gold_reconciliation
""")

assert {r.metric_name for r in dashboard_summary.collect()} == {
    "alerts",
    "supporting_transactions",
    "dq_checks_with_failures",
    "reconciliation_steps",
}
show_df(dashboard_summary.orderBy("metric_name"))

## Step 11 - Lakeflow / Jobs Thinking as a Runnable Plan

This notebook does not create Databricks Jobs or Lakeflow pipelines. Instead, it creates the task plan you should be able to explain before productionizing the notebook.

**Code-cell explanation:**

- Purpose: translate notebook cells into a production workflow plan.
- Inputs: no new source data; this is a structured task plan for the pipeline you just ran.
- Walkthrough: creates a task dependency table that maps ingestion, standardization, DQ, rule execution, reconciliation, publishing, and monitoring.
- Before running: predict the task order moves from bronze ingestion to monitoring, with DQ before rule execution.
- Failure meaning: if task dependencies are unclear, production orchestration will be fragile even when code cells work manually.

In [ ]:
job_task_plan = spark.createDataFrame([
    (1, "ingest_bronze", "Lakeflow Connect or ADF/Fabric landing task", "source checks and manifest"),
    (2, "standardize_silver", "Spark SQL or PySpark transformation", "schema, type, and key normalization checks"),
    (3, "run_dq", "Lakeflow expectations or explicit DQ queries", "exception tables and DQ summary"),
    (4, "build_gold_rule_input", "PySpark or Spark SQL rule-ready table", "row-count reconciliation"),
    (5, "execute_rule", "Databricks job task", "alerts and supporting transactions"),
    (6, "publish_evidence", "Databricks SQL / BI / audit pack task", "manifest, reconciliation, and sign-off artifacts"),
], ["task_order", "task_name", "databricks_pattern", "evidence_output"])

assert job_task_plan.count() == 6
show_df(job_task_plan.orderBy("task_order"))

## Step 12 - ML / Analytics Feature Readiness

ML in AML/TM should usually start as decision support. This feature table is not a model; it is a governed, explainable input that could support prioritization or false-positive analysis.

**Code-cell explanation:**

- Purpose: show how rule-ready data can become explainable feature-ready analytics without becoming an uncontrolled model.
- Inputs: `gold_rule_input`, `alerts`, and run metadata.
- Walkthrough: builds customer-level features and flags whether the customer produced an alert in this run.
- Before running: predict two customer feature rows and one alerted customer.
- Failure meaning: feature counts that do not reconcile to rule inputs can create leakage, drift, or misleading model evidence.

In [ ]:
customer_features = (
    gold_rule_input.groupBy("customer_id")
    .agg(
        F.count("*").alias("posted_wire_count"),
        F.sum("amount_cad").alias("posted_wire_amount_cad"),
        F.sum(F.when(F.col("risk_level") == "HIGH", 1).otherwise(0)).alias("high_risk_wire_count"),
        F.sum(F.when(F.col("risk_level") == "HIGH", F.col("amount_cad")).otherwise(F.lit(0))).alias("high_risk_wire_amount_cad"),
    )
)

alert_labels = alerts.select("customer_id").withColumn("alert_generated", F.lit(1))
feature_ready = customer_features.join(alert_labels, on="customer_id", how="left").fillna({"alert_generated": 0})

assert feature_ready.count() == 2
assert_set("feature customers", [r.customer_id for r in feature_ready.select("customer_id").collect()], ["c1", "c2"])
show_df(feature_ready.orderBy("customer_id"))

## Step 13 - Performance and Debugging Hooks

On Databricks, pair `explain` output with the Spark UI. Look for join strategy, shuffles, filters pushed before joins, skewed keys, and unnecessary wide transformations.

**Code-cell explanation:**

- Purpose: teach where to start when a Spark job is slow or unexpectedly expensive.
- Inputs: `gold_rule_input` and `alerts`.
- Walkthrough: prints the explain plan and partition counts so the learner can inspect joins, scans, shuffles, and task distribution.
- Before running: predict this cell produces diagnostic output, not business output.
- Failure meaning: inability to inspect plans or partitions makes performance tuning guesswork.

In [ ]:
print("Logical and physical plan for gold_rule_input:")
gold_rule_input.explain(True)

debug_counts = spark.createDataFrame([
    ("partitions_gold_rule_input", gold_rule_input.rdd.getNumPartitions()),
    ("partitions_alerts", alerts.rdd.getNumPartitions()),
], ["debug_metric", "debug_value"])
show_df(debug_counts)

## Step 14 - Tech Stack Micro-Lab: Same Rule in Spark SQL and PySpark

This cell group belongs here instead of Markdown because PySpark, Python, and Spark SQL learning should be runnable from a notebook. The goal is to prove that Spark SQL and PySpark can express the same AML/TM rule and produce the same expected rows.

Scenario: identify customers whose June 2022 posted transaction total is at least 10,000 CAD.

**Code-cell explanation:**

- Purpose: create a tiny independent dataset for a SQL-versus-PySpark comparison lab.
- Inputs: five in-memory transaction rows created in this cell.
- Walkthrough: defines an explicit schema, creates the micro DataFrame, registers a temp view, displays input rows, and asserts the input count.
- Before running: predict 5 input rows and no alert result yet; this cell only creates lab data.
- Failure meaning: if the setup count is wrong, the SQL and PySpark comparison later has no reliable baseline.

In [ ]:
micro_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("amount_cad", T.DoubleType(), False),
])

micro_rows = [
    ("T001", "C001", "2022-06-01", "POSTED", 6000.0),
    ("T002", "C001", "2022-06-10", "POSTED", 4500.0),
    ("T003", "C002", "2022-06-12", "POSTED", 3000.0),
    ("T004", "C002", "2022-06-20", "REVERSED", 9000.0),
    ("T005", "C003", "2022-07-01", "POSTED", 20000.0),
]

tech_stack_micro_transactions = spark.createDataFrame(micro_rows, micro_schema)
tech_stack_micro_transactions.createOrReplaceTempView("tech_stack_micro_transactions")

show_df(tech_stack_micro_transactions.orderBy("transaction_id"))
print("Expected input count: 5")
assert tech_stack_micro_transactions.count() == 5

### Step 14A - Run the Spark SQL Version

Expected output: one customer, `C001`, with June posted amount `10500.0`.

**Code-cell explanation:**

- Purpose: express the high-value June posted activity rule in Spark SQL.
- Inputs: `tech_stack_micro_transactions` temp view.
- Walkthrough: filters posted June rows, groups by customer, applies a 10000 CAD threshold, and orders results for deterministic display.
- Before running: predict only `C001` qualifies with `10500.0` CAD.
- Failure meaning: if `C002` or `C003` appears, date/status/threshold logic is wrong.

In [ ]:
tech_stack_sql_result = spark.sql("""
SELECT
    customer_id,
    ROUND(SUM(amount_cad), 2) AS june_posted_amount_cad
FROM tech_stack_micro_transactions
WHERE status = 'POSTED'
  AND transaction_date >= '2022-06-01'
  AND transaction_date < '2022-07-01'
GROUP BY customer_id
HAVING SUM(amount_cad) >= 10000
ORDER BY customer_id
""")

show_df(tech_stack_sql_result)

### Step 14B - Run the PySpark DataFrame Version

This should match the Spark SQL result exactly.

**Code-cell explanation:**

- Purpose: implement the same high-value rule using PySpark DataFrame transformations.
- Inputs: `tech_stack_micro_transactions` DataFrame.
- Walkthrough: applies the same filters, grouping, sum, threshold, and deterministic ordering as the SQL version.
- Before running: predict the PySpark output exactly matches the SQL output.
- Failure meaning: a mismatch means the two APIs are not representing the same business logic.

In [ ]:
tech_stack_pyspark_result = (
    tech_stack_micro_transactions
    .filter(
        (F.col("status") == "POSTED")
        & (F.col("transaction_date") >= "2022-06-01")
        & (F.col("transaction_date") < "2022-07-01")
    )
    .groupBy("customer_id")
    .agg(F.round(F.sum("amount_cad"), 2).alias("june_posted_amount_cad"))
    .filter(F.col("june_posted_amount_cad") >= 10000)
    .orderBy("customer_id")
)

show_df(tech_stack_pyspark_result)

### Step 14C - Validate SQL and PySpark Match

The assertions are the real learning contract: both implementations must produce the same expected business result.

**Code-cell explanation:**

- Purpose: make the micro-lab self-checking instead of relying on visual inspection.
- Inputs: `tech_stack_sql_result` and `tech_stack_pyspark_result`.
- Walkthrough: collects tiny result sets, compares both to the expected row, then compares SQL and PySpark to each other.
- Before running: predict all assertions pass and `tech_stack_micro_lab_status` becomes `PASS`.
- Failure meaning: assertion failure is the signal to debug business logic before scaling the pattern.

In [ ]:
tech_stack_sql_rows = [row.asDict() for row in tech_stack_sql_result.collect()]
tech_stack_pyspark_rows = [row.asDict() for row in tech_stack_pyspark_result.collect()]
tech_stack_expected_rows = [{"customer_id": "C001", "june_posted_amount_cad": 10500.0}]

assert tech_stack_sql_rows == tech_stack_expected_rows, tech_stack_sql_rows
assert tech_stack_pyspark_rows == tech_stack_expected_rows, tech_stack_pyspark_rows
assert tech_stack_sql_rows == tech_stack_pyspark_rows

tech_stack_micro_lab_status = "PASS"
print("SQL and PySpark rule outputs match expected result.")

## Step 15 - Final Validation Scorecard

If this cell passes, the notebook ran end to end and produced the expected learning artifacts.

**Code-cell explanation:**

- Purpose: give the main Databricks learning flow a single final pass/fail summary.
- Inputs: the main flow outputs and the tech-stack micro-lab status.
- Walkthrough: checks expected counts for bronze, silver, DQ, gold, alerts, supporting records, features, and the micro-lab.
- Before running: predict every row in the scorecard is `PASS`.
- Failure meaning: a failed scorecard item points to the exact stage that needs investigation.

In [ ]:
scorecard = spark.createDataFrame([
    ("bronze_count", "PASS" if bronze_transactions.count() == 8 else "FAIL"),
    ("silver_count", "PASS" if silver_transactions.count() == 8 else "FAIL"),
    ("orphan_visible", "PASS" if orphan_accounts.count() == 1 else "FAIL"),
    ("gold_rule_input_count", "PASS" if gold_rule_input.count() == 3 else "FAIL"),
    ("alert_count", "PASS" if alerts.count() == 1 else "FAIL"),
    ("supporting_transaction_count", "PASS" if supporting_transactions.count() == 2 else "FAIL"),
    ("feature_ready_count", "PASS" if feature_ready.count() == 2 else "FAIL"),
    ("tech_stack_micro_lab", "PASS" if tech_stack_micro_lab_status == "PASS" else "FAIL"),
], ["validation_name", "test_status"])

assert scorecard.filter(F.col("test_status") != "PASS").count() == 0
show_df(scorecard.orderBy("validation_name"))
print("Databricks one-stop notebook validation passed.")

## Closed-Book Drills

1. Change `rule_threshold_cad` to `120`. Predict the alert count before rerunning.
2. Change `t6` account from `a9` to `a2`. Predict the orphan count, gold input count, and alert count.
3. Change `t2` country from `IR` to `CA`. Predict `high_risk_supporting_tx` and `alerts`.
4. Explain which notebook cells would become Databricks Jobs tasks.
5. Explain which outputs should become governed Delta tables in Unity Catalog.
6. Explain which validation checks belong in Lakeflow expectations, explicit DQ tables, and CI.
7. In Step 14, change the micro-lab threshold from `10000` to `11000`. Predict whether `C001` still appears before rerunning.

## Appendix A - Consolidated Focused PySpark DataFrame Practice

This appendix consolidates the former focused PySpark notebook content. It stays inside the one-stop notebook so PySpark practice has one runnable home.

## Step 0 - Bootstrap

Expected setup: `transactions_raw` has 8 rows, `accounts` has 4 rows, and `country_risk` has 3 rows.

**Code-cell explanation:**

- Purpose: create the focused PySpark practice dataset inside the same canonical notebook.
- Inputs: in-memory synthetic transactions, accounts, and country-risk rows.
- Walkthrough: imports Spark helpers, creates explicit schemas/DataFrames, defines a set assertion helper, and validates source counts.
- Before running: predict 8 transactions, 4 accounts, and 3 country-risk rows.
- Failure meaning: if bootstrap fails, do not continue; later PySpark examples depend on these exact rows.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("aml-notebook-pyspark-basics").getOrCreate()

def assert_set(name, actual_rows, expected_rows):
    actual = set(actual_rows)
    expected = set(expected_rows)
    assert actual == expected, f"{name}: expected {expected}, got {actual}"

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("account_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("amount_cad", T.StringType(), False),
    T.StructField("transaction_type", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("country_code", T.StringType(), True),
])

transactions_raw = spark.createDataFrame([
    ("t1", "a1", "2022-06-01", "60.00", "WIRE", "POSTED", "IR"),
    ("t2", "a1", "2022-06-03", "50.00", "WIRE", "POSTED", "IR"),
    ("t3", "a1", "2022-06-05", "10.00", "CARD", "POSTED", "CA"),
    ("t4", "a2", "2022-06-02", "200.00", "WIRE", "POSTED", "CA"),
    ("t5", "a3", "2022-06-02", "20.00", "WIRE", "REVERSED", "IR"),
    ("t6", "a9", "2022-06-02", "80.00", "WIRE", "POSTED", "IR"),
    ("t7", "a2", "2022-07-01", "300.00", "WIRE", "POSTED", "IR"),
    ("t8", "a4", "2022-06-10", "100.00", "CASH", "POSTED", None),
], schema=transaction_schema)

accounts = spark.createDataFrame([
    ("a1", "c1", "ACTIVE", "CHECKING"),
    ("a2", "c2", "ACTIVE", "CHECKING"),
    ("a3", "c3", "ACTIVE", "SAVINGS"),
    ("a4", "c4", "CLOSED", "CHECKING"),
], ["account_id", "customer_id", "account_status", "product_type"])

country_risk = spark.createDataFrame([
    ("IR", "HIGH"),
    ("CA", "LOW"),
    ("US", "LOW"),
], ["country_code", "risk_level"])

assert transactions_raw.count() == 8
assert accounts.count() == 4
assert country_risk.count() == 3
print("Bootstrap validation passed.")

## Step 1 - Normalize Types

Spark reads the raw date and amount as strings here. Cast them before doing date filters or numeric thresholds.

**Code-cell explanation:**

- Purpose: show why raw strings must become typed business columns before filtering or thresholding.
- Inputs: `transactions_raw` from the focused PySpark bootstrap.
- Walkthrough: converts date strings to dates, amount strings to decimals, and standardizes account/country keys.
- Before running: predict 8 rows remain and the date/amount dtypes change.
- Failure meaning: type issues here represent source-format or schema-contract problems.

In [ ]:
transactions = (
    transactions_raw
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
    .withColumn("amount_cad", F.col("amount_cad").cast("decimal(18,2)"))
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("country_code", F.upper(F.trim("country_code")))
)

assert transactions.count() == 8
assert dict(transactions.dtypes)["transaction_date"] == "date"
assert dict(transactions.dtypes)["amount_cad"] == "decimal(18,2)"
transactions.orderBy("transaction_id").show(truncate=False)

## Step 2 - Filter Basics

Predict the row ids before running each filter. The important null lesson: `country_code != 'CA'` does not return null country rows.

**Code-cell explanation:**

- Purpose: teach row survival, date-window filtering, and null behavior.
- Inputs: normalized `transactions`.
- Walkthrough: creates posted-wire, June, missing-country, not-Canada, and not-Canada-or-missing subsets with assertions.
- Before running: predict `posted_wires` has 5 rows, June has 7 rows, and null country `t8` appears only when explicitly included.
- Failure meaning: wrong rows here usually mean misunderstanding SQL/Spark null semantics or half-open date windows.

In [ ]:
posted_wires = transactions.filter(F.col("status") == "POSTED").filter(F.col("transaction_type") == "WIRE")
june_transactions = transactions.filter(
    (F.col("transaction_date") >= F.lit("2022-06-01"))
    & (F.col("transaction_date") < F.lit("2022-07-01"))
)
missing_country = transactions.filter(F.col("country_code").isNull())
not_ca = transactions.filter(F.col("country_code") != "CA")
not_ca_or_missing = transactions.filter((F.col("country_code") != "CA") | F.col("country_code").isNull())

assert posted_wires.count() == 5
assert june_transactions.count() == 7
assert_set("missing_country", [r.transaction_id for r in missing_country.select("transaction_id").collect()], ["t8"])
assert "t8" not in {r.transaction_id for r in not_ca.select("transaction_id").collect()}
assert "t8" in {r.transaction_id for r in not_ca_or_missing.select("transaction_id").collect()}

posted_wires.orderBy("transaction_id").show(truncate=False)

## Step 3 - Joins, DQ, and Row Loss

The inner join hides orphan transaction `t6`. The left anti join makes the DQ exception explicit.

**Code-cell explanation:**

- Purpose: show how different join types affect row survival and DQ visibility.
- Inputs: `transactions`, `accounts`, and `country_risk`.
- Walkthrough: compares inner, left, left-anti, and left-semi joins, then identifies high-risk transactions.
- Before running: predict inner join has 7 rows and orphan transaction `t6` is visible through the left-anti join.
- Failure meaning: if orphan rows disappear without evidence, production rules can undercount risk silently.

In [ ]:
tx_with_accounts_inner = transactions.join(accounts, on="account_id", how="inner")
tx_with_accounts_left = transactions.join(accounts, on="account_id", how="left")
orphan_accounts = transactions.join(accounts, on="account_id", how="left_anti")
valid_account_transactions = transactions.join(accounts, on="account_id", how="left_semi")
tx_with_risk = transactions.join(country_risk, on="country_code", how="left")
high_risk_tx = tx_with_risk.filter(F.col("risk_level") == "HIGH")

assert tx_with_accounts_inner.count() == 7
assert tx_with_accounts_left.count() == 8
assert_set("orphan_accounts", [r.transaction_id for r in orphan_accounts.select("transaction_id").collect()], ["t6"])
assert valid_account_transactions.count() == 7
assert_set("high_risk_tx", [r.transaction_id for r in high_risk_tx.select("transaction_id").collect()], ["t1", "t2", "t5", "t6", "t7"])

orphan_accounts.show(truncate=False)

## Step 4 - Window Function

`row_number` lets you pick the latest transaction per account using a deterministic tie-breaker.

**Code-cell explanation:**

- Purpose: teach deterministic latest-row selection.
- Inputs: normalized `transactions`.
- Walkthrough: partitions by account, orders by date and transaction ID, assigns `row_number`, and keeps one row per account.
- Before running: predict the latest transaction IDs are `t3`, `t7`, `t5`, `t8`, and `t6` by account.
- Failure meaning: missing tie-breakers can make latest-row logic unstable across runs.

In [ ]:
latest_window = Window.partitionBy("account_id").orderBy(
    F.col("transaction_date").desc(),
    F.col("transaction_id").desc(),
)
latest_by_account = transactions.withColumn("rn", F.row_number().over(latest_window)).filter(F.col("rn") == 1).drop("rn")

latest_pairs = {(r.account_id, r.transaction_id) for r in latest_by_account.select("account_id", "transaction_id").collect()}
assert latest_pairs == {("a1", "t3"), ("a2", "t7"), ("a3", "t5"), ("a4", "t8"), ("a9", "t6")}
latest_by_account.orderBy("account_id").show(truncate=False)

## Step 5 - Build an Explainable AML/TM Alert

Rule idea: for June 2022, find posted WIRE transactions from high-risk countries, join to valid accounts, group by customer, and alert when total amount is greater than 100 CAD.

**Code-cell explanation:**

- Purpose: build a focused PySpark alert from filters, joins, aggregation, thresholding, and evidence linkage.
- Inputs: normalized transactions, account reference data, and country-risk reference data.
- Walkthrough: filters eligible June posted wires, keeps valid account/customer rows, keeps high-risk countries, aggregates by customer, creates deterministic alert metadata, and links supporting transactions.
- Before running: predict one alert for `c1` supported by `t1` and `t2`.
- Failure meaning: if alert count changes, debug filters, joins, risk mapping, and threshold boundaries in that order.

In [ ]:
june_posted_wires = (
    transactions.filter(F.col("status") == "POSTED")
    .filter(F.col("transaction_type") == "WIRE")
    .filter((F.col("transaction_date") >= F.lit("2022-06-01")) & (F.col("transaction_date") < F.lit("2022-07-01")))
)
valid_customer_tx = june_posted_wires.join(accounts, on="account_id", how="inner")
high_risk_customer_tx = valid_customer_tx.join(country_risk, on="country_code", how="inner").filter(F.col("risk_level") == "HIGH")
customer_totals = high_risk_customer_tx.groupBy("customer_id").agg(
    F.sum("amount_cad").alias("observed_amount_cad"),
    F.count("*").alias("supporting_transaction_count"),
)
alerts = (
    customer_totals.filter(F.col("observed_amount_cad") > F.lit(100))
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
    .withColumn("processing_month", F.lit("2022-06"))
    .withColumn("alert_key", F.sha2(F.concat_ws("|", "rule_id", "rule_version", "processing_month", "customer_id"), 256))
)
supporting_transactions = high_risk_customer_tx.join(alerts.select("alert_key", "customer_id"), on="customer_id", how="inner")

assert alerts.count() == 1
assert_set("alert customers", [r.customer_id for r in alerts.select("customer_id").collect()], ["c1"])
assert_set("supporting transaction ids", [r.transaction_id for r in supporting_transactions.select("transaction_id").collect()], ["t1", "t2"])

alerts.select("alert_key", "customer_id", "observed_amount_cad", "supporting_transaction_count").show(truncate=False)
supporting_transactions.select("alert_key", "transaction_id", "customer_id", "amount_cad", "country_code", "risk_level").orderBy("transaction_id").show(truncate=False)

## Step 6 - Reconciliation

A learning notebook should end with evidence. These counts explain where rows moved, where rows dropped, and why one alert was generated.

**Code-cell explanation:**

- Purpose: end the PySpark appendix with stage counts that explain row movement.
- Inputs: focused PySpark DataFrames from the appendix.
- Walkthrough: creates a reconciliation table, compares actual counts to expected counts, and prints a pass message.
- Before running: predict the expected-count dictionary matches actual counts exactly.
- Failure meaning: a mismatch tells you which transformation changed row movement and needs debugging.

In [ ]:
reconciliation = spark.createDataFrame([
    ("transactions", transactions.count()),
    ("posted_wires", posted_wires.count()),
    ("june_posted_wires", june_posted_wires.count()),
    ("valid_customer_tx", valid_customer_tx.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("high_risk_customer_tx", high_risk_customer_tx.count()),
    ("alerts", alerts.count()),
], ["step_name", "row_count"])

expected_counts = {
    "transactions": 8,
    "posted_wires": 5,
    "june_posted_wires": 4,
    "valid_customer_tx": 3,
    "orphan_accounts": 1,
    "high_risk_customer_tx": 2,
    "alerts": 1,
}
actual_counts = {r.step_name: r.row_count for r in reconciliation.collect()}
assert actual_counts == expected_counts, f"Expected {expected_counts}, got {actual_counts}"
reconciliation.show(truncate=False)
print("Notebook validation passed.")

## Closed-Book Drill

Before rerunning, change `t2` from `IR` to `CA`. Predict the new `high_risk_customer_tx`, `customer_totals`, and `alerts` counts.

## Appendix B - Consolidated Focused Spark SQL Practice

This appendix consolidates the former focused Spark SQL notebook content. It keeps Spark SQL practice in the same canonical notebook while still running through Python `spark.sql` cells.

## Step 0 - Bootstrap Temp Views

Expected setup: `transactions` has 8 rows, `accounts` has 4 rows, and `country_risk` has 3 rows.

**Code-cell explanation:**

- Purpose: create focused Spark SQL temp views inside the canonical notebook.
- Inputs: in-memory SQL `VALUES` rows for transactions, accounts, and country risk.
- Walkthrough: uses `spark.sql` to create temp views and validates their counts.
- Before running: predict 8 transactions, 4 accounts, and 3 country-risk rows.
- Failure meaning: if temp view creation fails, SQL examples later have no stable input contract.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('aml-notebook-spark-sql-basics').getOrCreate()

spark.sql('''
CREATE OR REPLACE TEMP VIEW transactions AS
SELECT * FROM VALUES
  ('t1', 'a1', DATE '2022-06-01', CAST(60.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t2', 'a1', DATE '2022-06-03', CAST(50.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t3', 'a1', DATE '2022-06-05', CAST(10.00 AS DECIMAL(18,2)),  'CARD', 'POSTED',   'CA'),
  ('t4', 'a2', DATE '2022-06-02', CAST(200.00 AS DECIMAL(18,2)), 'WIRE', 'POSTED',   'CA'),
  ('t5', 'a3', DATE '2022-06-02', CAST(20.00 AS DECIMAL(18,2)),  'WIRE', 'REVERSED', 'IR'),
  ('t6', 'a9', DATE '2022-06-02', CAST(80.00 AS DECIMAL(18,2)),  'WIRE', 'POSTED',   'IR'),
  ('t7', 'a2', DATE '2022-07-01', CAST(300.00 AS DECIMAL(18,2)), 'WIRE', 'POSTED',   'IR'),
  ('t8', 'a4', DATE '2022-06-10', CAST(100.00 AS DECIMAL(18,2)), 'CASH', 'POSTED',   NULL)
AS transactions(transaction_id, account_id, transaction_date, amount_cad, transaction_type, status, country_code)
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW accounts AS
SELECT * FROM VALUES
  ('a1', 'c1', 'ACTIVE', 'CHECKING'),
  ('a2', 'c2', 'ACTIVE', 'CHECKING'),
  ('a3', 'c3', 'ACTIVE', 'SAVINGS'),
  ('a4', 'c4', 'CLOSED', 'CHECKING')
AS accounts(account_id, customer_id, account_status, product_type)
''')

spark.sql('''
CREATE OR REPLACE TEMP VIEW country_risk AS
SELECT * FROM VALUES
  ('IR', 'HIGH'),
  ('CA', 'LOW'),
  ('US', 'LOW')
AS country_risk(country_code, risk_level)
''')

assert spark.table('transactions').count() == 8
assert spark.table('accounts').count() == 4
assert spark.table('country_risk').count() == 3
print('Bootstrap validation passed.')

## Step 1 - SELECT, WHERE, and Date Boundaries

Use half-open date windows for month filters: `>= 2022-06-01` and `< 2022-07-01`.

**Code-cell explanation:**

- Purpose: teach SQL row selection and half-open date windows.
- Inputs: `transactions` temp view from the SQL bootstrap.
- Walkthrough: selects posted wires and separately selects all June transactions, then validates counts.
- Before running: predict 5 posted wires and 7 June transactions.
- Failure meaning: wrong counts usually mean a date-boundary or eligibility-filter error.

In [ ]:
posted_wires = spark.sql('''
SELECT transaction_id, account_id, transaction_date, amount_cad, country_code
FROM transactions
WHERE status = 'POSTED'
  AND transaction_type = 'WIRE'
ORDER BY transaction_id
''')

june_transactions = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE transaction_date >= DATE '2022-06-01'
  AND transaction_date < DATE '2022-07-01'
''')

assert posted_wires.count() == 5
assert june_transactions.count() == 7
posted_wires.show(truncate=False)

## Step 2 - Null Handling

`country_code <> 'CA'` does not return nulls. You must include `OR country_code IS NULL` when missing values are part of the expected result.

**Code-cell explanation:**

- Purpose: make SQL null comparison behavior visible.
- Inputs: `transactions` temp view.
- Walkthrough: compares `country_code <> 'CA'` with an explicit `OR country_code IS NULL` version.
- Before running: predict `t8` is excluded from `not_ca` but included in `not_ca_or_missing`.
- Failure meaning: mishandling nulls can silently remove records from monitoring populations.

In [ ]:
not_ca = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE country_code <> 'CA'
''')

not_ca_or_missing = spark.sql('''
SELECT transaction_id
FROM transactions
WHERE country_code <> 'CA'
   OR country_code IS NULL
''')

not_ca_ids = {row.transaction_id for row in not_ca.collect()}
not_ca_or_missing_ids = {row.transaction_id for row in not_ca_or_missing.collect()}
assert 't8' not in not_ca_ids
assert 't8' in not_ca_or_missing_ids
not_ca_or_missing.orderBy('transaction_id').show(truncate=False)

## Step 3 - Aggregation and HAVING

After `GROUP BY`, the grain is no longer transaction. Here it becomes one row per account.

**Code-cell explanation:**

- Purpose: show how `GROUP BY` changes result grain and how `HAVING` filters groups.
- Inputs: `transactions` temp view.
- Walkthrough: groups by account, calculates counts and totals, and keeps accounts with totals over 100 CAD.
- Before running: predict account IDs `a1` and `a2` qualify.
- Failure meaning: using row-level filters when group-level filters are needed can produce wrong alert candidates.

In [ ]:
account_totals = spark.sql('''
SELECT
  account_id,
  COUNT(*) AS txn_count,
  SUM(amount_cad) AS total_amount_cad
FROM transactions
GROUP BY account_id
HAVING SUM(amount_cad) > 100
ORDER BY account_id
''')

assert {row.account_id for row in account_totals.collect()} == {'a1', 'a2'}
account_totals.show(truncate=False)

## Step 4 - Joins and DQ Exceptions

The left anti join is the cleanest way to show transactions that have no matching account.

**Code-cell explanation:**

- Purpose: show SQL join behavior and how to surface orphan records.
- Inputs: `transactions` and `accounts` temp views.
- Walkthrough: compares an inner join to a left anti join and validates orphan transaction `t6`.
- Before running: predict 7 inner-joined rows and one orphan row.
- Failure meaning: if orphan records are not counted, DQ defects can become hidden alert-volume differences.

In [ ]:
inner_joined = spark.sql('''
SELECT t.transaction_id, t.account_id, a.customer_id
FROM transactions t
JOIN accounts a
  ON t.account_id = a.account_id
''')

orphan_accounts = spark.sql('''
SELECT t.*
FROM transactions t
LEFT ANTI JOIN accounts a
  ON t.account_id = a.account_id
''')

assert inner_joined.count() == 7
assert {row.transaction_id for row in orphan_accounts.collect()} == {'t6'}
orphan_accounts.show(truncate=False)

## Step 5 - Window Function

A deterministic tie-breaker prevents latest-row logic from changing between runs.

**Code-cell explanation:**

- Purpose: teach SQL window logic for deterministic latest-row selection.
- Inputs: `transactions` temp view.
- Walkthrough: ranks rows by account using transaction date and transaction ID, then keeps `rn = 1`.
- Before running: predict one latest row per account, including orphan account `a9`.
- Failure meaning: missing deterministic order can create unstable results during reruns.

In [ ]:
latest_by_account = spark.sql('''
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY account_id
      ORDER BY transaction_date DESC, transaction_id DESC
    ) AS rn
  FROM transactions
)
SELECT account_id, transaction_id, transaction_date, amount_cad
FROM ranked
WHERE rn = 1
ORDER BY account_id
''')

expected_latest = {('a1', 't3'), ('a2', 't7'), ('a3', 't5'), ('a4', 't8'), ('a9', 't6')}
actual_latest = {(row.account_id, row.transaction_id) for row in latest_by_account.collect()}
assert actual_latest == expected_latest
latest_by_account.show(truncate=False)

## Step 6 - Build an Alert Query

This query keeps the rule explainable by preserving customer totals and supporting transactions.

**Code-cell explanation:**

- Purpose: build the AML/TM alert entirely in Spark SQL CTEs.
- Inputs: `transactions`, `accounts`, and `country_risk` temp views.
- Walkthrough: filters eligible transactions, joins references, keeps high-risk rows, aggregates by customer, creates deterministic alert keys, and registers `alerts` for later evidence joins.
- Before running: predict one alert for customer `c1`.
- Failure meaning: if the alert differs from the PySpark version, compare each CTE row count to isolate the defect.

In [ ]:
alerts = spark.sql('''
WITH june_posted_wires AS (
  SELECT *
  FROM transactions
  WHERE status = 'POSTED'
    AND transaction_type = 'WIRE'
    AND transaction_date >= DATE '2022-06-01'
    AND transaction_date < DATE '2022-07-01'
),
valid_customer_tx AS (
  SELECT t.*, a.customer_id
  FROM june_posted_wires t
  JOIN accounts a
    ON t.account_id = a.account_id
),
high_risk_customer_tx AS (
  SELECT t.*, r.risk_level
  FROM valid_customer_tx t
  JOIN country_risk r
    ON t.country_code = r.country_code
  WHERE r.risk_level = 'HIGH'
),
customer_totals AS (
  SELECT
    customer_id,
    SUM(amount_cad) AS observed_amount_cad,
    COUNT(*) AS supporting_transaction_count
  FROM high_risk_customer_tx
  GROUP BY customer_id
)
SELECT
  SHA2(CONCAT_WS('|', 'TM_HIGH_RISK_WIRE_001', '1.0.0', '2022-06', customer_id), 256) AS alert_key,
  'TM_HIGH_RISK_WIRE_001' AS rule_id,
  '1.0.0' AS rule_version,
  '2022-06' AS processing_month,
  customer_id,
  observed_amount_cad,
  supporting_transaction_count
FROM customer_totals
WHERE observed_amount_cad > 100
''')

alerts.createOrReplaceTempView('alerts')
assert alerts.count() == 1
assert {row.customer_id for row in alerts.collect()} == {'c1'}
alerts.show(truncate=False)

## Step 7 - Supporting Transactions and Reconciliation

An alert is weak without the records and counts that prove it.

**Code-cell explanation:**

- Purpose: prove the SQL alert has supporting transaction evidence and reconciled counts.
- Inputs: SQL temp views and the `alerts` view from Step 6.
- Walkthrough: rebuilds the high-risk support population, joins it to alerts, builds reconciliation counts, and validates expected support rows.
- Before running: predict supporting transactions `t1` and `t2`, one alert, and one orphan account.
- Failure meaning: missing support rows or count mismatches mean the alert is not audit-ready.

In [ ]:
supporting_transactions = spark.sql('''
WITH june_posted_wires AS (
  SELECT *
  FROM transactions
  WHERE status = 'POSTED'
    AND transaction_type = 'WIRE'
    AND transaction_date >= DATE '2022-06-01'
    AND transaction_date < DATE '2022-07-01'
),
high_risk_customer_tx AS (
  SELECT t.*, a.customer_id, r.risk_level
  FROM june_posted_wires t
  JOIN accounts a
    ON t.account_id = a.account_id
  JOIN country_risk r
    ON t.country_code = r.country_code
  WHERE r.risk_level = 'HIGH'
)
SELECT
  a.alert_key,
  h.transaction_id,
  h.customer_id,
  h.account_id,
  h.amount_cad,
  h.country_code,
  h.risk_level
FROM high_risk_customer_tx h
JOIN alerts a
  ON h.customer_id = a.customer_id
ORDER BY h.transaction_id
''')

reconciliation = spark.createDataFrame([
    ('transactions', spark.table('transactions').count()),
    ('posted_wires', posted_wires.count()),
    ('orphan_accounts', orphan_accounts.count()),
    ('supporting_transactions', supporting_transactions.count()),
    ('alerts', alerts.count()),
], ['step_name', 'row_count'])

assert {row.transaction_id for row in supporting_transactions.collect()} == {'t1', 't2'}
expected_counts = {
    'transactions': 8,
    'posted_wires': 5,
    'orphan_accounts': 1,
    'supporting_transactions': 2,
    'alerts': 1,
}
actual_counts = {row.step_name: row.row_count for row in reconciliation.collect()}
assert actual_counts == expected_counts, f'Expected {expected_counts}, got {actual_counts}'

supporting_transactions.show(truncate=False)
reconciliation.show(truncate=False)
print('Notebook validation passed.')

## Closed-Book Drill

Rewrite the alert query without looking. Then explain which step can drop rows, which step can create duplicates, and which counts you would put into a production reconciliation report.

## Appendix C - WHERE vs HAVING and PySpark Filter Placement Micro-Lab

This appendix turns a classic interview topic into runnable proof: `WHERE` filters transaction rows before `GROUP BY`, `HAVING` filters aggregated groups after it, and PySpark has no `HAVING` keyword at all - the position of `.filter()` relative to `.groupBy().agg()` decides which gate you wrote.

It reuses the `transactions` temp view from **Appendix B Step 0**, so run that bootstrap first.

Theory guide: [`docs/spark/where-having-filter-placement.md`](../../../docs/spark/where-having-filter-placement.md)

## Step 1 - Two Filter Gates in One Query (Spark SQL)

The rule: alert any account whose POSTED WIRE transactions total more than 100 CAD. "Posted wires" is the row gate (`WHERE`); "total more than 100" is the group gate (`HAVING`).

**Code-cell explanation:**

- Purpose: prove that dropping the row gate changes evidence totals even when the surviving account set looks identical.
- Inputs: `transactions` temp view from Appendix B Step 0.
- Walkthrough: runs the governed query (`WHERE` posted wires plus `HAVING` total over 100) and an ungoverned variant with no `WHERE`, then compares per-account totals.
- Before running: predict both queries return accounts `a1` and `a2`, but `a1` totals 110.00 with the row gate and 120.00 without it because CARD transaction `t3` leaks into the group.
- Failure meaning: if the totals match, the row gate is not running where you think; an account-set reconciliation alone would have passed and hidden the wrong evidence.

In [ ]:
correct_alerts = spark.sql('''
SELECT
  account_id,
  COUNT(*) AS txn_count,
  SUM(amount_cad) AS total_amount_cad
FROM transactions
WHERE status = 'POSTED'
  AND transaction_type = 'WIRE'
GROUP BY account_id
HAVING SUM(amount_cad) > 100
ORDER BY account_id
''')

no_row_filter = spark.sql('''
SELECT
  account_id,
  COUNT(*) AS txn_count,
  SUM(amount_cad) AS total_amount_cad
FROM transactions
GROUP BY account_id
HAVING SUM(amount_cad) > 100
ORDER BY account_id
''')

correct_totals = {row.account_id: float(row.total_amount_cad) for row in correct_alerts.collect()}
inflated_totals = {row.account_id: float(row.total_amount_cad) for row in no_row_filter.collect()}

assert correct_totals == {'a1': 110.0, 'a2': 500.0}
assert inflated_totals == {'a1': 120.0, 'a2': 500.0}
assert set(correct_totals) == set(inflated_totals)

correct_alerts.show(truncate=False)
print('Same alert accounts, different evidence: a1 is 110.00 posted-wire CAD, not 120.00.')

## Step 2 - The Structuring Trap: a Row Filter Is Not a Group Filter

Putting the amount threshold in `WHERE` filters individual transactions, not account totals. Account `a1` wires 60 and 50: each row stays under 100, only the total crosses it. That split pattern is structuring - exactly what the rule exists to catch.

**Code-cell explanation:**

- Purpose: show that moving the threshold from the group gate to the row gate silently changes which accounts can ever alert.
- Inputs: `transactions` temp view.
- Walkthrough: runs the wrong version (`WHERE amount_cad > 100`) and the right version (`HAVING SUM(amount_cad) > 100`) and compares surviving account sets.
- Before running: predict the wrong version returns only `a2` (rows t4 and t7 pass) while the right version returns `a1` and `a2`.
- Failure meaning: if the sets match, the threshold is not where you think; in production this defect produces no error - structuring accounts just stop alerting.

In [ ]:
row_threshold = spark.sql('''
SELECT account_id, SUM(amount_cad) AS total_amount_cad
FROM transactions
WHERE status = 'POSTED'
  AND transaction_type = 'WIRE'
  AND amount_cad > 100
GROUP BY account_id
ORDER BY account_id
''')

group_threshold = spark.sql('''
SELECT account_id, SUM(amount_cad) AS total_amount_cad
FROM transactions
WHERE status = 'POSTED'
  AND transaction_type = 'WIRE'
GROUP BY account_id
HAVING SUM(amount_cad) > 100
ORDER BY account_id
''')

row_threshold_accounts = {row.account_id for row in row_threshold.collect()}
group_threshold_accounts = {row.account_id for row in group_threshold.collect()}

assert row_threshold_accounts == {'a2'}
assert group_threshold_accounts == {'a1', 'a2'}
assert group_threshold_accounts - row_threshold_accounts == {'a1'}

print('Structuring account missed by the row-level threshold:', group_threshold_accounts - row_threshold_accounts)

## Step 3 - PySpark Has No HAVING: Placement Decides Meaning

PySpark exposes one filtering method (`filter`, alias `where`). A filter **before** `groupBy` is the `WHERE` gate on rows; a filter **after** `agg` is the `HAVING` gate on the aggregated DataFrame, referencing the aggregate's alias.

**Code-cell explanation:**

- Purpose: express both gates in PySpark by position alone and reconcile the result against the Spark SQL version from Step 1.
- Inputs: `transactions` temp view read as a DataFrame, plus `correct_alerts` from Step 1.
- Walkthrough: builds the posted-wire row gate before `groupBy`, the total threshold after `agg`, plus the wrong-placement variant from Step 2, then asserts PySpark and SQL agree row for row.
- Before running: predict the post-agg filter keeps `a1` (110.00) and `a2` (500.00), the wrong placement keeps only `a2`, and the SQL reconciliation passes.
- Failure meaning: a mismatch means a filter crossed the aggregation boundary - same schema, same types, different rule.

In [ ]:
from pyspark.sql import functions as F

tx = spark.table('transactions')

posted_wires = tx.filter((F.col('status') == 'POSTED') & (F.col('transaction_type') == 'WIRE'))

account_totals = (
    posted_wires
    .groupBy('account_id')
    .agg(
        F.count('*').alias('txn_count'),
        F.sum('amount_cad').alias('total_amount_cad'),
    )
)

having_style = account_totals.filter(F.col('total_amount_cad') > 100)

wrong_placement = (
    posted_wires
    .filter(F.col('amount_cad') > 100)
    .groupBy('account_id')
    .agg(F.sum('amount_cad').alias('total_amount_cad'))
)

pyspark_totals = {row.account_id: float(row.total_amount_cad) for row in having_style.collect()}
sql_totals = {row.account_id: float(row.total_amount_cad) for row in correct_alerts.collect()}

assert pyspark_totals == sql_totals == {'a1': 110.0, 'a2': 500.0}
assert {row.account_id for row in wrong_placement.collect()} == {'a2'}

having_style.orderBy('account_id').show(truncate=False)
print('PySpark filter-after-agg reconciles with SQL WHERE plus HAVING.')

## Closed-Book Drill

Without looking, write the posted-wire structuring rule twice: once in Spark SQL with `WHERE` plus `HAVING`, and once in PySpark with two `.filter()` calls in the correct positions. Then explain which account a row-level `amount_cad > 100` filter loses and why, and why the Step 1 account-set reconciliation passed while the evidence was still wrong. Model answers are in [`docs/spark/where-having-filter-placement.md`](../../../docs/spark/where-having-filter-placement.md) sections 5, 6, and 11.

## Appendix D - Amount Drift Across Bronze/Silver/Gold Micro-Lab

When a total amount differs between medallion layers, the amount was dropped, duplicated, transformed, or reclassified - never "just changed." This appendix proves the main factors with assertions: silent cast loss, dedupe and quarantine accounting, FX conversion as an approved difference, and join explosion.

This appendix is self-contained: it builds its own tiny bronze batch.

Theory and the full factor catalog: [`docs/04-data-quality-reconciliation-defect-management.md`](../../../docs/04-data-quality-reconciliation-defect-management.md), section 11.

## Step 0 - Bootstrap a Tiny Bronze Batch

Expected setup: 7 raw bronze rows (including one European-formatted amount, one empty amount, and one exact duplicate), 1 FX rate, and 5 account-ownership rows where account `a4` has two ownership periods.

**Code-cell explanation:**

- Purpose: create the smallest batch that can demonstrate every major amount-drift factor.
- Inputs: in-memory synthetic bronze amounts (as strings, like a real landing zone), an FX rate table, and effective-dated account ownership.
- Walkthrough: builds the three DataFrames and validates their counts.
- Before running: predict 7 bronze rows, 1 FX rate, 5 ownership rows.
- Failure meaning: if the bootstrap fails, the later conservation assertions have no stable input contract.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('aml-notebook-amount-drift').getOrCreate()

bronze_amounts = spark.createDataFrame([
    ('d1', 'a1', '2022-06-15', '100.10', 'CAD'),
    ('d2', 'a1', '2022-06-15', '1,250.40', 'CAD'),
    ('d3', 'a2', '2022-06-15', '', 'CAD'),
    ('d4', 'a2', '2022-06-15', '200.20', 'CAD'),
    ('d5', 'a3', '2022-06-15', '300.00', 'CAD'),
    ('d5', 'a3', '2022-06-15', '300.00', 'CAD'),
    ('d6', 'a4', '2022-06-15', '100.00', 'USD'),
], ['transaction_id', 'account_id', 'transaction_date', 'amount_raw', 'currency_code'])
bronze_amounts = bronze_amounts.withColumn('transaction_date', F.to_date('transaction_date'))

fx_rates = spark.createDataFrame([
    ('USD', 'CAD', '1.3567'),
], ['from_currency', 'to_currency', 'rate']).withColumn('rate', F.col('rate').cast('decimal(10,4)'))

account_owners = spark.createDataFrame([
    ('a1', 'c1', '2020-01-01', '9999-12-31'),
    ('a2', 'c2', '2020-01-01', '9999-12-31'),
    ('a3', 'c3', '2020-01-01', '9999-12-31'),
    ('a4', 'c4', '2020-01-01', '2022-01-01'),
    ('a4', 'c5', '2022-01-01', '9999-12-31'),
], ['account_id', 'customer_id', 'effective_start', 'effective_end'])
account_owners = (
    account_owners
    .withColumn('effective_start', F.to_date('effective_start'))
    .withColumn('effective_end', F.to_date('effective_end'))
)

assert bronze_amounts.count() == 7
assert fx_rates.count() == 1
assert account_owners.count() == 5
print('Amount-drift bootstrap validation passed.')

## Step 1 - Bronze to Silver: the Cast That Loses Money Silently

`try_cast` turns unparseable amounts into nulls, and `SUM` skips nulls - so one formatting character can move 1250.40 out of the total with no error. With ANSI mode on, a plain `CAST` would instead fail the whole run: decide deliberately which failure surface you want, and catch the silent one with a cast-failure DQ check.

**Code-cell explanation:**

- Purpose: show that a naive cast shrinks the total silently and that the cast-failure DQ check (raw present, typed null) catches exactly the lost rows.
- Inputs: `bronze_amounts` from Step 0.
- Walkthrough: casts naively, computes the shrunken total, separates cast failures (`d2`) from genuinely missing amounts (`d3`), then re-parses with the comma handled and recovers the true total.
- Before running: predict the naive total is 1000.30 and the parsed total is 2250.70, a 1250.40 difference from a single comma.
- Failure meaning: if the totals match, the cast did not behave as expected on this runtime; check the ANSI mode setting.

In [ ]:
naive_silver = bronze_amounts.withColumn('amount', F.expr('try_cast(amount_raw AS DECIMAL(18,2))'))

cast_failures = naive_silver.filter(F.col('amount').isNull() & (F.trim('amount_raw') != ''))
missing_amounts = naive_silver.filter(F.trim('amount_raw') == '')

naive_total = naive_silver.agg(F.sum('amount').alias('t')).collect()[0]['t']
assert float(naive_total) == 1000.30
assert [r.transaction_id for r in cast_failures.collect()] == ['d2']
assert [r.transaction_id for r in missing_amounts.collect()] == ['d3']

parsed_silver = bronze_amounts.withColumn(
    'amount', F.expr("try_cast(replace(amount_raw, ',', '') AS DECIMAL(18,2))")
)
parsed_total = parsed_silver.agg(F.sum('amount').alias('t')).collect()[0]['t']
assert float(parsed_total) == 2250.70

print('Naive total:', naive_total, '| parsed total:', parsed_total)
print('One formatting character silently moved 1250.40 out of the naive total.')

## Step 2 - Dedupe and Quarantine: Conservation of Amount

Removing the duplicate `d5` and quarantining the missing-amount row `d3` both lower the silver population - correctly. The control is the conservation identity: the upstream total must equal the downstream total plus every excluded amount, with reasons.

**Code-cell explanation:**

- Purpose: prove the conservation identity `parsed_total = silver_total + duplicate_removed` and account for every row: 7 bronze = 5 silver + 1 quarantined + 1 duplicate removed.
- Inputs: `parsed_silver` from Step 1.
- Walkthrough: routes null-amount rows to quarantine, deduplicates on `transaction_id`, computes the removed-duplicate amount from copy counts, and asserts the identity.
- Before running: predict silver total 1950.70, duplicate-removed amount 300.00, quarantine 1 row.
- Failure meaning: an identity mismatch means an amount left the pipeline without a reason - exactly the defect this case study exists to catch.

In [ ]:
quarantined = parsed_silver.filter(F.col('amount').isNull())
silver_clean = parsed_silver.filter(F.col('amount').isNotNull()).dropDuplicates(['transaction_id'])

silver_total = silver_clean.agg(F.sum('amount').alias('t')).collect()[0]['t']

dup_extra = (
    parsed_silver.filter(F.col('amount').isNotNull())
    .groupBy('transaction_id', 'amount')
    .agg(F.count('*').alias('copies'))
    .withColumn('extra_amount', F.col('amount') * (F.col('copies') - 1))
)
duplicate_removed = dup_extra.agg(F.sum('extra_amount').alias('t')).collect()[0]['t']

assert float(silver_total) == 1950.70
assert float(duplicate_removed) == 300.00
assert float(parsed_total) == float(silver_total) + float(duplicate_removed)
assert bronze_amounts.count() == silver_clean.count() + quarantined.count() + 1

print('Conservation holds: 2250.70 = 1950.70 (silver) + 300.00 (duplicate removed); 1 row quarantined with a reason.')

## Step 3 - Silver to Gold: FX Is an Approved Difference, Join Explosion Is a Defect

Two factors change the gold total in opposite governance categories. Currency conversion changes values **with evidence** (rate, rate date, source amount) - an approved difference. A reference join missing its effective-date bounds duplicates rows, changing the total **without changing any value** - a defect.

**Code-cell explanation:**

- Purpose: convert USD to CAD as a documented transformation, then contrast a correctly bounded ownership join with an unbounded one that double-counts.
- Inputs: `silver_clean` from Step 2, `fx_rates` and `account_owners` from Step 0.
- Walkthrough: applies the FX rate to `d6` (100.00 USD -> 135.67 CAD), asserts the explained uplift of 35.67, then joins ownership with and without effective-date predicates and compares totals and counts.
- Before running: predict the converted total 1986.37, the bad-join total 2122.04 (one extra copy of 135.67), and one extra row in the bad join.
- Failure meaning: if the two joins agree, the duplicate-ownership trap did not fire; check the effective dates on account `a4`.

In [ ]:
silver_cad = (
    silver_clean
    .join(F.broadcast(fx_rates), silver_clean.currency_code == fx_rates.from_currency, 'left')
    .withColumn(
        'amount_cad',
        F.when(F.col('rate').isNotNull(), (F.col('amount') * F.col('rate')).cast('decimal(18,2)'))
        .otherwise(F.col('amount'))
    )
    .select('transaction_id', 'account_id', 'transaction_date', 'amount_cad')
)

converted_total = silver_cad.agg(F.sum('amount_cad').alias('t')).collect()[0]['t']
assert float(converted_total) == 1986.37
assert round(float(converted_total) - float(silver_total), 2) == 35.67

gold_good = silver_cad.alias('t').join(
    account_owners.alias('o'),
    (F.col('t.account_id') == F.col('o.account_id'))
    & (F.col('t.transaction_date') >= F.col('o.effective_start'))
    & (F.col('t.transaction_date') < F.col('o.effective_end')),
    'inner',
)
gold_bad = silver_cad.join(account_owners, 'account_id', 'inner')

good_total = gold_good.agg(F.sum('amount_cad').alias('t')).collect()[0]['t']
bad_total = gold_bad.agg(F.sum('amount_cad').alias('t')).collect()[0]['t']

assert float(good_total) == 1986.37
assert float(bad_total) == 2122.04
assert gold_bad.count() == gold_good.count() + 1

print('FX uplift +35.67 is approved with rate evidence; the unbounded join added 135.67 with no new transactions - a defect.')

## Closed-Book Drill

Without looking, name the amount-drift factor for each symptom: total off by exactly x100; pennies off and growing with volume; counts match but totals differ; totals inflate with rising row counts and no new transactions; difference appears only on rerun. Then state the conservation-of-amount identity and which two pieces of evidence make the FX difference in Step 3 acceptable. Model answers are in [`docs/04-data-quality-reconciliation-defect-management.md`](../../../docs/04-data-quality-reconciliation-defect-management.md) sections 11 and 12.

## Final Consolidated Notebook Check

If every validation cell above passed, the one-stop notebook now covers the Databricks modernization flow, focused PySpark DataFrame basics, focused Spark SQL basics, DQ, reconciliation, alert evidence, and tech-stack micro-lab practice.

**Code-cell explanation:**

- Purpose: mark the end of the consolidated notebook after all main-flow and appendix validations have run.
- Inputs: no new data; this cell is a final visible success marker.
- Walkthrough: prints a single success message after all prior assertions have passed.
- Before running: confirm every earlier validation cell has completed without error.
- Failure meaning: if execution stopped before this cell, the last failed assertion identifies the learning section to debug.

In [ ]:
print("Consolidated Databricks/Spark/PySpark/SQL notebook validation passed.")